# 청킹 방식 4세트 비교

절 제목을 청크에 붙이면 검색이 좋아지는지, 표 청크에 기관명·사업명을 붙이는 것이 도움이 되는지 실측했다.

| 세트 | 청크 수 | 구성 |
|---|---|---|
| 기존 | 10,222 | 본문만. 표는 구분자 없이 뭉개짐, 이미지 없음 |
| 추출 | 17,382 | 본문 + 표 + 이미지 (이슈 #84 결과) |
| 제목 | 18,518 | 추출 + 절 경계 존중 + 본문 청크에 절 제목 |
| 표머리 | 18,518 | 제목 + 표 청크 첫 줄에 `[기관명 · 사업명]` |

네 세트를 같은 임베딩 모델(`text-embedding-3-small`)로 각각 색인해 조건을 맞췄다.
생성은 `gpt-5-nano`가 비결정적이라 문항마다 3회 반복해 평균을 냈다.


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

if Path.cwd().name == "notebook":
    os.chdir("..")

RESULT_PATH = Path.home() / "vision_cmp" / "chunk_comparison.json"
ORDER = ["기존", "추출", "추출+제목", "추출+제목+표머리말"]
SHORT = {"기존": "기존", "추출": "추출", "추출+제목": "제목", "추출+제목+표머리말": "표머리"}

results = pd.DataFrame(json.loads(RESULT_PATH.read_text(encoding="utf-8")))
results["rate"] = results["keyword_hit"] / results["keyword_total"]
print(f"{len(results)}행 · 세트 {results['set'].nunique()}개 · 문항 {results['question_id'].nunique()}개")

156행 · 세트 4개 · 문항 13개


## 1. 종합 지표

In [2]:
summary = results.groupby(["question_id", "set"]).agg(
    유형=("type", "first"),
    키워드=("keyword_hit", "mean"),
    총=("keyword_total", "first"),
    정답문서=("doc_hit", "first"),
    표=("table_chunks", "first"),
    이미지=("image_chunks", "first"),
).reset_index()

overall = pd.DataFrame([
    {
        "세트": SHORT[name],
        "키워드 적중률": f"{results[results['set'] == name]['rate'].mean() * 100:.1f}%",
        "정답문서(5중)": round(summary[summary["set"] == name]["정답문서"].mean(), 2),
        "표 점유(5중)": round(summary[summary["set"] == name]["표"].mean(), 1),
    }
    for name in ORDER
]).set_index("세트")
display(overall)

,키워드 적중률,정답문서(5중),표 점유(5중)
세트,,,
기존,60.3%,2.69,0.0
추출,62.0%,3.23,2.6
제목,74.4%,3.31,2.4
표머리,48.7%,3.69,3.8


`제목`이 키워드 적중률에서 가장 좋고, `표머리`는 정답 문서를 가장 잘 찾지만 답을 못 한다.
문서를 맞게 찾고도 그 안에서 엉뚱한 표가 올라오기 때문이다.

## 2. 문항별 결과

In [3]:
pivot = summary.pivot(index="question_id", columns="set")
detail = pd.DataFrame({"유형": pivot[("유형", "기존")]})
for name in ORDER:
    detail[f"키 {SHORT[name]}"] = pivot[("키워드", name)].round(1)
for name in ORDER:
    detail[f"문서 {SHORT[name]}"] = pivot[("정답문서", name)]
display(detail)

,유형,키 기존,키 추출,키 제목,키 표머리,문서 기존,문서 추출,문서 제목,문서 표머리
question_id,,,,,,,,,
Q01,single,3.0,3.0,3.0,0.0,2,3,3,5
Q02,numeric,2.0,1.3,2.0,2.0,2,4,3,5
Q03,single,3.0,3.0,3.0,3.0,3,4,4,5
Q04,single,3.0,3.0,3.0,3.0,5,4,4,5
Q05,table,0.0,6.0,6.0,0.0,0,1,1,0
Q06,table,3.0,3.0,3.0,3.0,2,4,4,4
Q07,multi_document,2.0,2.0,4.0,2.0,2,3,4,4
Q08,unsupported,1.0,1.0,0.3,0.0,0,0,0,0
Q09,follow_up,2.0,0.0,2.0,2.0,5,4,5,5


### 단계별로 무엇이 달라졌나

In [4]:
steps = []
for before, after in zip(ORDER, ORDER[1:]):
    steps.append({
        "변화": f"{SHORT[before]} → {SHORT[after]}",
        "키워드": f"{(results[results['set'] == after]['rate'].mean() - results[results['set'] == before]['rate'].mean()) * 100:+.1f}%p",
        "정답문서": f"{summary[summary['set'] == after]['정답문서'].mean() - summary[summary['set'] == before]['정답문서'].mean():+.2f}",
        "표 점유": f"{summary[summary['set'] == after]['표'].mean() - summary[summary['set'] == before]['표'].mean():+.1f}",
    })
display(pd.DataFrame(steps).set_index("변화"))

,키워드,정답문서,표 점유
변화,,,
기존 → 추출,+1.7%p,+0.54,+2.6
추출 → 제목,+12.4%p,+0.08,-0.2
제목 → 표머리,-25.6%p,+0.38,+1.5


## 3. 절 제목의 효과

키워드 적중률이 62.0%에서 74.4%로 올랐다. 표·이미지를 넣어 얻은 폭(+1.7%p)보다 크다.

- **Q07(다문서 비교)** 2.0/4 → 4.0/4. 기존과 추출 양쪽에서 움직이지 않던 문항이다
- **Q09(후속 질문)** 0.0/2 → 2.0/2. 표 청크가 본문을 밀어내 깨졌던 것이 복구됐다
- **Q13(이미지)** 1.7/3 → 2.7/3

Q09가 특히 의미가 있다. 이슈 #84에서 `top_k` 조정이 필요하다고 Retrieval에 넘기려던 문제인데,
검색 설정을 바꾸지 않고 데이터 쪽에서 해소됐다.

In [5]:
def escape(text):
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def compare(question_id, sets):
    part = results[results["question_id"] == question_id]
    item = part.iloc[0]
    panels = ""
    for name in sets:
        one = part[(part["set"] == name) & (part["trial"] == 1)].iloc[0]
        mean_hit = part[part["set"] == name]["keyword_hit"].mean()
        panels += f"""
        <div style="flex:1; min-width:0; border:1px solid #999; border-radius:6px; padding:10px;">
          <div style="font-weight:bold;">{SHORT[name]}</div>
          <div style="font-size:12px; margin-bottom:6px;">
            키워드 {mean_hit:.1f}/{one['keyword_total']} · 정답문서 {one['doc_hit']} ·
            표 {one['table_chunks']} · 이미지 {one['image_chunks']}
          </div>
          <pre style="white-space:pre-wrap; font-size:12px; margin:0;">{escape(one['answer'])}</pre>
        </div>"""
    display(HTML(f"""
    <div style="margin:20px 0;">
      <div style="font-weight:bold;">{question_id} · {item['type']}</div>
      <div style="font-size:13px; margin:4px 0;">{escape(item['question'])}</div>
      <div style="font-size:12px; margin-bottom:8px;">정답 키워드: {escape(', '.join(item['keywords']))}</div>
      <div style="display:flex; gap:12px; align-items:flex-start;">{panels}</div>
    </div>"""))


for question_id in ("Q07", "Q09"):
    compare(question_id, ["추출", "추출+제목"])

## 4. 표 머리말을 채택하지 않은 이유

PR #88 리뷰에서 표 청크에도 기관명·사업명을 붙이자는 제안이 있었다.
표 청크 12,892개 중 기관명이 텍스트에 있는 것은 360개(3%)뿐이어서 지적은 사실이었고, 붙인 뒤 100%가 됐다.

리뷰에서 함께 제시된 두 축으로 측정했다.

In [6]:
axis = pd.DataFrame({
    "정답문서(5중)": [
        round(summary[summary["set"] == "추출+제목"]["정답문서"].mean(), 2),
        round(summary[summary["set"] == "추출+제목+표머리말"]["정답문서"].mean(), 2),
    ],
    "표 점유(5중)": [
        round(summary[summary["set"] == "추출+제목"]["표"].mean(), 1),
        round(summary[summary["set"] == "추출+제목+표머리말"]["표"].mean(), 1),
    ],
    "키워드 적중률": [
        f"{results[results['set'] == '추출+제목']['rate'].mean() * 100:.1f}%",
        f"{results[results['set'] == '추출+제목+표머리말']['rate'].mean() * 100:.1f}%",
    ],
}, index=["제목", "표머리"])
display(axis)

print("(1) 기관명 언급 질문에서 검색이 개선되는가 -> 그렇다. 정답문서 4세트 중 최고")
print("(2) 문서 안에서 구분이 흐려지는가        -> 그렇다. 표 점유가 늘고 키워드가 기존보다도 낮아짐")

,정답문서(5중),표 점유(5중),키워드 적중률
제목,3.31,2.4,74.4%
표머리,3.69,3.8,48.7%


(1) 기관명 언급 질문에서 검색이 개선되는가 -> 그렇다. 정답문서 4세트 중 최고
(2) 문서 안에서 구분이 흐려지는가        -> 그렇다. 표 점유가 늘고 키워드가 기존보다도 낮아짐


In [7]:
for question_id in ("Q01", "Q05"):
    compare(question_id, ["추출+제목", "추출+제목+표머리말"])

문서는 맞게 찾지만 그 안에서 엉뚱한 표가 올라온다.
한 문서의 표 전부에 같은 문구가 붙어 문서 간 구분은 좋아지고 문서 내 구분이 무너진 것이다.
짧은 표일수록 40자가 넘는 머리말이 실제 내용을 덮는다.

## 결론

**추출 + 절 제목**을 채택한다.

```yaml
chunking:
  use_section_titles: true       # 채택
  use_table_headline: false      # 기각
  max_section_title_length: 40
```

표 머리말은 문서 식별력이 오르는 것이 분명하므로, 메타데이터 필터나 하이브리드 검색 쪽에서 활용할 여지는 남는다.

### 알려진 한계

- 본문 청크 5,333개 중 절 제목이 붙은 것은 2,240개(42%)다. 표지·목차처럼 제목이 없는 절이 있고,
  제목을 표가 아니라 일반 문단으로 만든 문서도 있다
- 절 길이 편차가 크다(중앙값 282자, 최대 45,805자). 긴 절은 절 안에서 800자로 다시 나눈다
- PDF 4건은 제목 표가 없어 기존 방식으로 청킹한다
- Q08(근거 없음 문항)은 키워드 지표의 의미가 약해 판단에서 제외했다
